<a href="https://colab.research.google.com/github/kushim2005/omniproject/blob/main/PDF_Parsing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install langchain-text-splitters

import fitz                        # PyMuPDF
import json
import os
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ─── CONFIG ────────────────────────────────────────────────────────────
PDF_PATH    = "/content/Application_of_Machine_Learning_to_Predict_Mental_Health_Disorders_and_Interpret_Feature_Importance.pdf"   # <-- change to your PDF path
OUTPUT_DIR  = "output"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "chunks.json")

CHUNK_SIZE    = 500    # characters per chunk
CHUNK_OVERLAP = 50     # overlap between chunks

# ─── SETUP ─────────────────────────────────────────────────────────────
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ─── STEP 1: Extract Text from PDF ─────────────────────────────────────
def extract_text(pdf_path: str) -> list:
    """
    Read every page of the PDF and extract raw text.
    Returns: list of { page, raw_text }
    """
    doc   = fitz.open(pdf_path)
    pages = []

    print(f"[INFO] Opened: {pdf_path}  ({len(doc)} pages)")

    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text()

        if text.strip():          # skip blank pages
            pages.append({
                "page": page_num + 1,
                "raw_text": text
            })
            print(f"  [EXTRACTED] Page {page_num + 1} — {len(text)} chars")

    doc.close()
    print(f"\n[INFO] Pages with text: {len(pages)}")
    return pages


# ─── STEP 2: Clean Text ─────────────────────────────────────────────────
def clean_text(text: str) -> str:
    """
    Remove extra whitespace, newlines, and unwanted characters.
    """
    text = re.sub(r'\n+',    ' ', text)    # newlines → space
    text = re.sub(r'\s+',    ' ', text)    # multiple spaces → single
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # remove non-ASCII
    return text.strip()


# ─── STEP 3: Split into Chunks ──────────────────────────────────────────
def chunk_text(pages: list, source_filename: str) -> list:
    """
    Split each page's text into chunks using LangChain splitter.
    Returns: list of { page, chunk_id, chunk, source }
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size    = CHUNK_SIZE,
        chunk_overlap = CHUNK_OVERLAP,
        separators    = ["\n\n", "\n", ". ", " ", ""]
    )

    all_chunks = []
    chunk_id   = 1

    for page_data in pages:
        cleaned = clean_text(page_data["raw_text"])
        splits  = splitter.split_text(cleaned)

        for split in splits:
            all_chunks.append({
                "chunk_id": chunk_id,
                "page":     page_data["page"],
                "chunk":    split,
                "source":   source_filename
            })
            chunk_id += 1

    print(f"[INFO] Total chunks created: {len(all_chunks)}")
    return all_chunks


# ─── STEP 4: Save Metadata as JSON ─────────────────────────────────────
def save_json(chunks: list, output_path: str):
    """Save all chunks with metadata to a JSON file."""
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(chunks, f, indent=2, ensure_ascii=False)
    print(f"[INFO] Saved -> {output_path}")


# ─── MAIN ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    source_name = os.path.basename(PDF_PATH)

    # Step 1 – Extract
    pages  = extract_text(PDF_PATH)

    # Step 2 & 3 – Clean + Chunk
    chunks = chunk_text(pages, source_name)

    # Step 4 – Save JSON
    save_json(chunks, OUTPUT_FILE)

    # Preview first chunk (matches image output format)
    if chunks:
        print("\n── Sample Output ────────────────────────────────────")
        sample = {
            "page":   chunks[0]["page"],
            "chunk":  chunks[0]["chunk"][:60] + "...",
            "source": chunks[0]["source"]
        }
        print(json.dumps(sample, indent=2))
        print("─────────────────────────────────────────────────────")

    print(f"\n[DONE] {len(chunks)} chunks saved to {OUTPUT_FILE}")

[INFO] Opened: /content/Application_of_Machine_Learning_to_Predict_Mental_Health_Disorders_and_Interpret_Feature_Importance.pdf  (5 pages)
  [EXTRACTED] Page 1 — 5556 chars
  [EXTRACTED] Page 2 — 5718 chars
  [EXTRACTED] Page 3 — 5186 chars
  [EXTRACTED] Page 4 — 3495 chars
  [EXTRACTED] Page 5 — 6045 chars

[INFO] Pages with text: 5
[INFO] Total chunks created: 63
[INFO] Saved -> output/chunks.json

── Sample Output ────────────────────────────────────
{
  "page": 1,
  "chunk": "Application of machine learning to predict mental health dis...",
  "source": "Application_of_Machine_Learning_to_Predict_Mental_Health_Disorders_and_Interpret_Feature_Importance.pdf"
}
─────────────────────────────────────────────────────

[DONE] 63 chunks saved to output/chunks.json
